In [1]:
# I learned the approach implemented in this notebook
# https://www.kaggle.com/code/yuriygreben/birdclef-26-onnx-perch-dual-ssms-vectorized-mlp
# will try to implemented some of the ideas of the notebook.

In [2]:
#%pip install -q --no-deps /kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

In [3]:
mode = 'train_offline'
run_build_cache = True

In [4]:
import librosa
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from pathlib import Path
if "offline" in mode:
    print('working offline')
    import tensorflow as tf
    import math
    import soundfile as sf
    import mlflow
    tf.config.set_visible_devices([], 'GPU')  # force CPU
elif "online" in mode or mode == 'submit':
    %pip install -q --no-deps /kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
import onnxruntime as ort

working offline


I0000 00:00:1780274307.336850   20618 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Download perch_v2_cpu and run it and cache the results

First, try to run ssm on the audio file in train_soundscapes folder


In [5]:
kaggle_onnx_path = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/perch_v2.onnx')
# dictionary to perch_v2 for different modes
onnx_path_dict = {
    'train_offline': Path('../models/perch_onnx/perch_v2.onnx'),
    'train_online': kaggle_onnx_path,
    'submit': kaggle_onnx_path,
    'test_submit': kaggle_onnx_path,
}
kaggle_base_path = Path('/kaggle/input/competitions/birdclef-2026')
base_path_dict = {
    'train_offline': Path('../data'),
    'train_online': kaggle_base_path,
    'submit': kaggle_base_path,
    'test_submit': kaggle_base_path,
}
onnx_path = onnx_path_dict[mode]
base_path = base_path_dict[mode]

In [6]:
session_option = ort.SessionOptions()
session_option.intra_op_num_threads = 4
onnx_session = ort.InferenceSession(onnx_path, session_options = session_option, providers = ['CPUExecutionProvider'])
onnx_ipt_name = onnx_session.get_inputs()[0].name
print(f'Onnx input ame: {onnx_ipt_name}')
onnx_opt_map = {o.name: i for i, o in enumerate(onnx_session.get_outputs())}
print(f'Onnx output maps: {onnx_opt_map}')

Onnx input ame: inputs
Onnx output maps: {'embedding': 0, 'spatial_embedding': 1, 'spectrogram': 2, 'label': 3}


### Build soundscapes cache


### Get the sites and the hours where and when sounds are recorded

Sites and the hours will be added as additional features along with the features obtained from perch-v2


In [7]:
import re
# train_soundscapes file names has site up to 20, so this pattern is used
# if it does not match, it means the files in test_sounscapes folder may have different patterns
filename_pattern = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(\w{3})_(\w{8})_(\w{6})\.ogg")
def get_site_month_hour(filename) -> tuple[str, int,int]:
    """
    Arg:
    filename: the filename of a file in soundscape folder that has site and hour information
    Return:
    A tuple with filename as string and site as an integer
    """
    _, site, date, hour = filename_pattern.match(filename).groups()

    return (site, int(date[4:6]), int(hour[:2]))

    
def build_file_name_arr(soundscapes_folder_path):
    return 0
    
get_site_month_hour('BC2026_Train_0001_S08_20250606_030007.ogg')


('S08', 6, 3)

In [8]:

train_soundscapes_label_df = pd.read_csv(base_path / 'train_soundscapes_labels.csv')
train_soundscapes_label_df['site'], train_soundscapes_label_df['month'], train_soundscapes_label_df['hour'] = zip(*train_soundscapes_label_df['filename'].apply(get_site_month_hour))

sites = sorted(train_soundscapes_label_df['site'].unique())
hours = sorted(train_soundscapes_label_df['hour'].unique())
months = sorted(train_soundscapes_label_df['month'].unique())
print(f'sites: {sites} \n hours: {hours}\n months: {months}')

sites: ['S03', 'S08', 'S09', 'S13', 'S15', 'S18', 'S19', 'S22', 'S23'] 
 hours: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
 months: [np.int64(1), np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(10), np.int64(11), np.int64(12)]


In [9]:
# apparently, each row in train_soundscapes_label_df is duplicated
print(f'Length of train_soundscapes_label_df before deduplicate: {len(train_soundscapes_label_df)}')
train_soundscapes_label_df = train_soundscapes_label_df.drop_duplicates()
print(f'Length of train_soundscapes_label_df after deduplicate: {len(train_soundscapes_label_df)}')

Length of train_soundscapes_label_df before deduplicate: 1478
Length of train_soundscapes_label_df after deduplicate: 739


In [10]:
train_soundscapes_label_df.head()

,filename,start,end,primary_label,site,month,hour
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380,S22,12,20
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380,S22,12,20
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380,S22,12,20
3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380,S22,12,20
4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380,S22,12,20


In [11]:

if "offline" in mode:
    perch_label_path = Path('../models/perch_onnx/labels.csv')
    
elif "online" in mode or mode == 'submit':
    perch_label_path = Path('/kaggle/input/datasets/dangthaigiang/perch-v2-cpu-onnx-head/labels.csv')

perch_df = pd.read_csv(perch_label_path)
# rename the column to match the column name in taxonomy.csv
perch_df.rename(columns={'inat2024_fsd50k': 'scientific_name'}, inplace=True)
perch_df.head()

,scientific_name
0,Abavorana luctuosa
1,Abeillia abeillei
2,Abroscopus albogularis
3,Abroscopus schisticeps
4,Abroscopus superciliaris


In [12]:
taxonomy_df = pd.read_csv(base_path / 'taxonomy.csv')
taxonomy_df.head()

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia


In [13]:
taxonomy_join_perch_df = taxonomy_df.merge(perch_df.rename_axis("perch_idx").reset_index(), on='scientific_name', how='left')
taxonomy_join_perch_df.head()


,primary_label,inat_taxon_id,scientific_name,common_name,class_name,perch_idx
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta,5743.0
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia,NaN
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia,7018.0
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia,NaN
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia,NaN


In [14]:
taxonomy_df.head()

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia


In [15]:

nan_mask = taxonomy_join_perch_df['perch_idx'].isna()
print(f"Number of nan in perch_idx: {nan_mask.sum()}")

Number of nan in perch_idx: 31


In [16]:
# fill NaN in 'perch_idx' with len(perch_df)-an unknown species
unknow_species_idx = len(perch_df)
taxonomy_join_perch_df['perch_idx'] = taxonomy_join_perch_df['perch_idx'].fillna(unknow_species_idx)
taxonomy_join_perch_df['perch_idx'] = taxonomy_join_perch_df['perch_idx'].astype(np.int32)
taxonomy_join_perch_df.head()

,primary_label,inat_taxon_id,scientific_name,common_name,class_name,perch_idx
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta,5743
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia,14795
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia,7018
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia,14795
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia,14795


In [17]:
submission_df = pd.read_csv(base_path / 'sample_submission.csv')
submission_df.head()

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [18]:
#labels of species in the submission file
submission_species_labels = submission_df.columns[1:].to_list()
# assert that species in submision are the same and are in the same order as primary_labels in taxonomy
assert(submission_species_labels == taxonomy_df['primary_label'].to_list())
"Mismatch between labels in submission and taxonomy "
#species that are in submision but and in perch

label_to_perch_idx = taxonomy_join_perch_df.set_index("primary_label")["perch_idx"]
species_in_perch_mask = taxonomy_join_perch_df['perch_idx'] != unknow_species_idx
species_not_in_perch_mask = ~species_in_perch_mask
#scientific names of species not in perch
#positions of species in perch (also in taxaxonomy) in the taxonomy table
species_in_perch_positions = taxonomy_join_perch_df[species_in_perch_mask].index.to_list()
#positions of species not in perch (but in taxaxonomy) in the taxonomy table
species_not_in_perch_positions = taxonomy_join_perch_df[species_not_in_perch_mask].index.to_list()
print(species_in_perch_positions)
print(species_not_in_perch_positions)
# for each species not in perch, returns list of scienticfic names of all species which are in the same genus 
# as the species not in perch



[0, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 55, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233]
[1, 3, 4, 23, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41

In [19]:
# rows of species that are not in perch
taxonomy_join_perch_df[species_not_in_perch_mask]

,primary_label,inat_taxon_id,scientific_name,common_name,class_name,perch_idx
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia,14795
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia,14795
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia,14795
23,25073,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,14795
30,47158son01,47158,Insect son01,Insect sonotype01,Insecta,14795
31,47158son02,47158,Insect son02,Insect sonotype02,Insecta,14795
32,47158son03,47158,Insect son03,Insect sonotype03,Insecta,14795
33,47158son04,47158,Insect son04,Insect sonotype04,Insecta,14795
34,47158son05,47158,Insect son05,Insect sonotype05,Insecta,14795
35,47158son06,47158,Insect son06,Insect sonotype06,Insecta,14795


In [20]:
# perch indices of species that are in taxonomy and in perch
perch_idx_species_in_perch = taxonomy_join_perch_df[species_in_perch_mask]['perch_idx'].to_list()
print(perch_idx_species_in_perch)

[5743, 7018, 4745, 7037, 7023, 7032, 7002, 7006, 7012, 7030, 10365, 10358, 10359, 11307, 10246, 12393, 12387, 12386, 12373, 11813, 4504, 4122, 10363, 9609, 427, 2158, 10376, 10512, 1591, 1601, 1602, 516, 4102, 4105, 11273, 12036, 13854, 7019, 10630, 1666, 11079, 6287, 8365, 3575, 8048, 3176, 13626, 11156, 1029, 4054, 8120, 4286, 1062, 1850, 12852, 13752, 8430, 7738, 11171, 14213, 8235, 13652, 460, 13655, 1309, 3181, 9351, 8155, 5580, 3290, 13255, 8049, 8965, 8957, 14072, 9568, 13439, 6445, 6915, 2166, 5537, 7503, 8483, 14206, 3134, 6264, 1055, 2630, 12257, 3673, 13448, 8460, 10505, 8956, 9978, 13955, 7095, 4860, 1050, 712, 9725, 726, 4531, 4514, 4635, 1057, 4365, 3144, 12583, 11783, 2438, 3028, 10916, 2761, 12196, 12707, 6423, 463, 9605, 13235, 5272, 13684, 4316, 9768, 6922, 13635, 6529, 3806, 13654, 3809, 5987, 1030, 2138, 3488, 5353, 2254, 7715, 8491, 12204, 14140, 11336, 2300, 3333, 5275, 894, 9979, 10847, 8494, 13740, 5296, 1173, 12642, 13685, 3273, 10462, 10003, 8418, 40, 7449, 11

In [21]:
# genus of species that are not in perch
not_in_perch_genera = taxonomy_join_perch_df[species_not_in_perch_mask]['scientific_name'].apply(lambda x: x.split(' ')[0]).unique()
not_in_perch_genera



array(['Caiman', 'Adenomera', 'Lysapsus', 'Chiasmocleis', 'Insect',
       'Sapajus', 'Mico'], dtype=object)

In [22]:
def get_genus_of_species(species):
    """
    Arg:
    species_not_in_perch: a string representing a species not in perch
    Return:
    a string representing the genus of the species
    """
    return species.split(' ')[0]
def get_perch_idx_of_species_of_same_genus(genus):
    """"
    Given a genus, return a list of all perch indices of species in the same genus 
    Args: 
    genus: a string representing a genus
    Return:
    a list of perch indices of species in the same genus 
    """
    genus_species = perch_df[perch_df['scientific_name'].str.contains(genus)]
    genus_species_idx = genus_species.index.tolist()
    return genus_species_idx
def get_perch_idx_of_species_of_same_genus_as_species_not_in_perch(species_not_in_perch):
    """
    Arg:
    species_not_in_perch: a string representing a species not in perch
    Return:
    a list of perch indices of species in the same genus
    """
    genus = get_genus_of_species(species_not_in_perch)
    return get_perch_idx_of_species_of_same_genus(genus)

# for each species not in perch, get the list of perch indices of species in the same genus
not_in_perch_idx_list = taxonomy_join_perch_df[species_not_in_perch_mask]['scientific_name'].apply(get_perch_idx_of_species_of_same_genus_as_species_not_in_perch)
for idx in not_in_perch_idx_list.index:
    print(f'{idx}: {not_in_perch_idx_list[idx]}')

1: [1953, 1954]
3: [143, 144, 145, 146, 147, 148, 149, 150, 151]
4: [7482, 7483]
23: [2611, 2612, 2613, 2614, 2615, 2616]
30: [6541]
31: [6541]
32: [6541]
33: [6541]
34: [6541]
35: [6541]
36: [6541]
37: [6541]
38: [6541]
39: [6541]
40: [6541]
41: [6541]
42: [6541]
43: [6541]
44: [6541]
45: [6541]
46: [6541]
47: [6541]
48: [6541]
49: [6541]
50: [6541]
51: [6541]
52: [6541]
53: [6541]
54: [6541]
56: [12272, 12273, 12274]
70: [8042]


In [23]:
SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: int) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    Arguments:
        path: path to .ogg file
        offset_sec: offset in seconds into the file
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
sample_path = base_path / 'train_soundscapes' / 'BC2026_Train_0001_S08_20250606_030007.ogg'
chunk = load_chunk(sample_path, offset_sec=0.0)
print('Shape:', chunk.shape)   # (160000,)
print('dtype:', chunk.dtype)   # float32
print('Range:', chunk.min(), chunk.max())

Shape: (160000,)
dtype: float32
Range: -0.19364518 0.19562061


In [24]:
taxonomy_df['primary_label'].values

array(['1161364', '116570', '1176823', '1491113', '1595929', '209233',
       '22930', '22956', '22961', '22967', '22973', '22983', '22985',
       '23150', '23154', '23158', '23176', '23724', '24279', '24285',
       '24287', '24321', '244024', '25073', '25092', '25214', '326272',
       '41970', '43435', '47144', '47158son01', '47158son02',
       '47158son03', '47158son04', '47158son05', '47158son06',
       '47158son07', '47158son08', '47158son09', '47158son10',
       '47158son11', '47158son12', '47158son13', '47158son14',
       '47158son15', '47158son16', '47158son17', '47158son18',
       '47158son19', '47158son20', '47158son21', '47158son22',
       '47158son23', '47158son24', '47158son25', '476521', '516975',
       '517063', '555123', '555145', '555146', '64898', '65377', '65380',
       '66971', '67107', '67252', '70711', '738183', '74113', '74580',
       '760266', 'ashgre1', 'astcra1', 'bafcur1', 'baffal1', 'banana',
       'barant1', 'batbel1', 'baymac', 'bbwduc', 'bcwfi

In [25]:
def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = waveform[np.newaxis, :]
    outs = onnx_session.run(None, {onnx_ipt_name: inp})
    # print(len(outs))
    emb  = outs[onnx_opt_map['embedding']].astype(np.float32)
    label_logits = outs[onnx_opt_map['label']]  # (1, 508)

    
    return emb, label_logits # (1536,)

embedding, label_logtis = extract_embedding(chunk)
print(f'Embedding shape: {embedding.shape}')
print(f'Label logits shape: {label_logtis.shape}')

Embedding shape: (1, 1536)
Label logits shape: (1, 14795)


In [26]:
train_soundscapes_label_df.columns

Index(['filename', 'start', 'end', 'primary_label', 'site', 'month', 'hour'], dtype='object')

In [27]:
# Building cache for train_soundscapes
# Only process the files in train_soundscapes_labels.csv

N_CLASSES = 234
N_WINDOWS = 12
EMBEDDING_DIMENSION = 1536
file_windows = train_soundscapes_label_df.groupby('filename').size()
full_windows_files = file_windows[file_windows == N_WINDOWS].index.to_list()
full_windows_files_mask = train_soundscapes_label_df['filename'].isin(full_windows_files)
# a sub df of train_soundscapes_label_df containing rows from 'full" 1 minute files
full_windows_files_df = (train_soundscapes_label_df[full_windows_files_mask]
                        .sort_values(['filename', 'end'])
                        .reset_index(drop=False))

filenames = (base_path / 'train_soundscapes').glob('*.ogg')
print(len(list(filenames)))
print(len(train_soundscapes_label_df))
full_windows_filenames = full_windows_files_df['filename'].unique()
train_rows_number = len(full_windows_filenames) * N_WINDOWS
def build_train_soundscapes_cache():
    
    row_ids = np.empty(train_rows_number, dtype=object)
    filenames = np.empty(train_rows_number, dtype=object)
    sites = np.empty(train_rows_number, dtype=object)
    months = np.empty(train_rows_number, dtype=object)
    hours = np.empty(train_rows_number, dtype=object)
    scores = np.empty((train_rows_number, N_CLASSES), dtype=np.float32)
    embeddings = np.empty((train_rows_number, EMBEDDING_DIMENSION), dtype=np.float32)
    stop_iteration = len(full_windows_filenames)
    for i in range(stop_iteration):
        train_soundscape_filename = full_windows_filenames[i]
        train_soundscape_path = base_path / 'train_soundscapes' / train_soundscape_filename
        site, month, hour = get_site_month_hour(train_soundscape_filename)
        for j in range(N_WINDOWS):
            off_set_sec = j*5
            chunk = load_chunk(train_soundscape_path, off_set_sec)
            embedding, label_logits = extract_embedding(chunk)
            row_ids[i*N_WINDOWS+j] = train_soundscape_filename[:-4]+'_'+str(off_set_sec+5)
            filenames[i*N_WINDOWS+j] = train_soundscape_filename
            sites[i*N_WINDOWS+j] = site
            months[i*N_WINDOWS+j] = month
            hours[i*N_WINDOWS+j] = hour
            scores[i*N_WINDOWS+j,species_in_perch_positions] = label_logits[0, perch_idx_species_in_perch]
            for taxonomy_idx in not_in_perch_idx_list.index:
                # take the max across all genus-mates per row
                # intuition: if the "max" genus-mate is present, the un-mapped species
                # is likely to present too.
                scores[i*N_WINDOWS+j, taxonomy_idx] = np.max(label_logits[0, not_in_perch_idx_list[taxonomy_idx]])
            embeddings[i*N_WINDOWS+j] = embedding
    meta_df = pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'month': months, 'hour': hours})   
    print(len(meta_df))
    META_CACHE_PATH = base_path / 'meta_train_soundscapes.parquet'
    meta_df.to_parquet(META_CACHE_PATH)
    TRAIN_SOUNDSCAPES_CACHE_PATH = base_path / "train_soundscapes_cache.npz"
    np.savez_compressed(
        TRAIN_SOUNDSCAPES_CACHE_PATH,
        embeddings=embeddings,
        scores=scores,
        primary_labels=np.array(taxonomy_df['primary_label'].values),
    )
if run_build_cache:
    build_train_soundscapes_cache()

10658
739
708


### Dealing with labels in train_soundscapes_labels.csv


In [28]:
def convert_labels_string_to_list(label_string):
    label_list = label_string.split(';')
    return label_list
    # return [label.strip() for label in label_list]
convert_labels_string_to_list('22967;22973;517063;trsowl')

['22967', '22973', '517063', 'trsowl']

In [29]:

# Scores only for files that are full, i.e, 1 minute files which have 12 window
Y_full_scores = np.zeros((train_rows_number, N_CLASSES), dtype=np.uint8)
PRIMARY_LABELS = taxonomy_df['primary_label'].values
label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
for i in range(train_rows_number):
    labels = full_windows_files_df['primary_label'].iloc[i].split(';')
    label_idx_list = [label_to_idx[label] for label in labels]
    Y_full_scores[i, label_idx_list] = 1



### Build model


### StandardSelectiveSSM
Slightly different from the protoSSM dual vectorized mlp notebook

In [ ]:
import torch.nn.functional as F
hyper_parameters = {
    'd_model': 128, # general size of matrixes that would flow between layers
    'd_state': 16, # number of states of SSM the larger the d_states the more stored hidden information
}
class StandardSelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state, d_conv_kernel=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=True)
        self.one_d_conv = nn.Conv1d(
            d_model, # in_channels - numper of input features
            d_model, # out_channels - number of output features
            kernel_size=d_conv_kernel, # how many consecutive timesteps the filter looks at once
            padding=d_conv_kernel-1, # zero-padding added to both ends of the sequence
            groups=d_model, # depthwise convolution
        )
        self.dt_project = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1) # (D,N) N=d_state, -1: keeps dimesion -1 unchange
        self.A_log = nn.Parameter(torch.log(A)) # log of A  
        self.B_project = nn.Linear(d_model, d_state, bias=False)
        self.C_project = nn.Linear(d_model, d_state, bias=False)
        self.D = nn.Parameter(torch.ones(d_model))  
        self.out_project = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B_size, T, D_size = x.shape # (B, T, D) (D= d_model) B = B_size = batch size
        x_and_res = self.in_proj(x) # (B, T, 2 * D) 
        x, res = x_and_res.chunk(2, dim=-1) # two (B, T, D)
        x = self.one_d_conv(x.transpose(1, 2))[:,:,:T].transpose(1, 2) # covolution time dimension, then transpose back
        x = F.silu(x) # (B, T, D)
        dt = self.dt_project(x) # (B, T, D)@(D,D) -> (B, T, D)
        dt = F.softplus(dt) # (B, T, D)
        A = -torch.exp(self.A_log) # (D, N) N=d_state
        B = self.B_project(x) # (B, T, N)  
        C = self.C_project(x) # (B, T, N) 
        h = torch.zeros(B_size, self.d_model, self.d_state, device=x.device) # (B, D, N)
        ys = []
        for t in range(T):
            # A (D, N) A[None] (1, D, N), dt (B, T, D) dt[:, t, :, None] (B, D, 1) 
            # A[None] * dt[:, t, :, None] broadcast -> (B, D, N)
            dA = torch.exp(A[None] * dt[:, t, :, None]) 
            # dt[:,t,:,Non] (B,D,1) * B (B, T, N) B[:,t,None,:] (B, 1, N) broadcast -> (B, D, N)
            dB = dt[:,t,:,None] * B[:,t,None,:]
            h = dA * h + dB * x[:,t,:,None] # (B, D, N) * (B, D, 1) -> (B, D, N) 
            y = (h * C[:, t, None, :]).sum(-1) # (B, D, N) * (B, 1, N) -> (B, D, N).sum(-1) -> (B,D)
            ys.append(y)
        y = torch.stack(ys, dim=1) # (B, T, D)
        y = y + (x * self.D[None, None, :]) # (B, T, D) D skip (over x)
        y = y * F.silu(res) # (B, T, D) gate with residual
        return self.out_project(y) 